# Tutoriel SimulacraBench

Nous utiliserons le schéma d'exemple
`data/sample.json` pour comprendre la forme de la tâche, ainsi que la raison plus technique d'être de la compétition. Voir `README.md` pour le contrat de soumission, la règle de score et le règlement.

In [1]:
import os
import sys
from pathlib import Path

# Ce carnet se trouve dans tutorials/, mais le harnais se trouve à la
# racine du dépôt, et tous les chemins ci-dessous -- config.yml,
# data/sample.json, _sandbox/ -- sont écrits relativement à cette racine. On la
# localise et on travaille depuis là, pour que le carnet fonctionne de la même
# façon que Jupyter ait été lancé dans ce dossier ou au-dessus.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "make_sandbox.py").exists()), None)
if ROOT is None:
    raise RuntimeError("exécutez ce carnet depuis un clone du dépôt")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

# Les mêmes modules que ceux qu'utilise score.py. Rien ici ne
# réimplémente l'évaluateur : quand vous notez quelque chose dans ce carnet,
# vous appelez le code qui vous note.
from make_sandbox import (ROLE_COLUMN, generated_items, load_config,
                          load_schema, options_for, write_sandbox)
from score import floored, load_frames, sample_rows
from score import score as grade

SEED = 0
PHASE = 1
config = load_config("config.yml")

pd.set_option("display.width", 200, "display.max_columns", 50)

---

## 1. La forme de la tâche

`make_sandbox.py` transforme un schéma en un jeu de données ayant exactement la
forme du vrai : mêmes colonnes, mêmes modalités, même logique de filtre. Les
distributions marginales et les dépendances sont inventées : un pipeline mis au
point ici sera transposable, un modèle ajusté ici ne le sera pas.

Il écrit `respondents.parquet` — chaque répondant, plus le `role` qui indique à
quoi il sert — et `schema.json`, qui est ce que reçoit votre `predict()`. Les
rôles sont fixés une seule fois, ici, et écrits sur disque. `score.py` les
consulte ; il ne partitionne jamais rien lui-même.

In [2]:
sample = load_schema("data/sample.json", config)
write_sandbox(sample, config, "_sandbox/sample", seed=SEED)

respondents = load_frames("_sandbox/sample", sample)
print(sample["dataset"]["description"])
print()

MEANING = {"TRAIN": "visible dans les deux phases",
           "DEV": "noté en phase 1, visible en phase 2",
           "TEST": "noté en phase 2"}
counts = respondents[ROLE_COLUMN].value_counts()
print(pd.DataFrame({"répondants": counts,
                    "signification": [MEANING[r] for r in counts.index]}).to_string())

A toy instrument, not a real survey. Ten items, few enough to print the whole schema and read it. It has one of everything the real schemas have: a frame block that is always visible, items that are scored, a gate chain two deep, and an EXCLUDE column the grader never shows anybody. The GIVEN block is deliberately the cheap half of a questionnaire -- the variables that already sit on a sampling frame, a census roster or another survey of the same households -- and the PREDICT block is the expensive half, the part that needs an enumerator and an interview. Use it to see the shape of the task; use the three real schemas to see whether a method works.

       répondants                        signification
role                                                  
TRAIN        8000         visible dans les deux phases
TEST         2100                      noté en phase 2
DEV          1900  noté en phase 1, visible en phase 2


### Ce que déclare le schéma

Chaque item porte quatre clés : la `question` telle qu'elle est posée, une
`class`, les `values` qu'il autorise, et un `gate` s'il n'est posé qu'à
certaines personnes.

Les trois classes résument toute la tâche. **`GIVEN`** est visible pour tout le
monde et n'est jamais noté. **`PREDICT`** est masqué pour les répondants
masqués, et chaque case vide est notée. **`EXCLUDE`** — identifiants, clés
d'enregistrement, texte libre — n'est jamais livré dans la table, donc filtrez
sur `class` plutôt que de supposer que le schéma et la table portent les mêmes
colonnes.

Dans cet instrument, la séparation oppose délibérément la moitié bon marché
d'un questionnaire à sa moitié coûteuse. Le bloc `GIVEN` est le genre de
variable qui figure déjà dans une base de sondage, un recensement ou une autre
enquête auprès des mêmes ménages : lieu de résidence, taille du ménage,
possession d'un téléphone. Le bloc `PREDICT` est ce qui exige un enquêteur et
un entretien. La partie 2 repose sur cette distinction.

In [3]:
rows = []
for name, rec in sample["items"].items():
    gate = rec.get("gate") or {}
    rows.append({"item": name,
                 "classe": rec["class"],
                 "K": len(options_for(sample, name)) if rec["values"] else 0,
                 "filtre": gate.get("parent", "-"),
                 "posé si": ", ".join(gate.get("observed_if", [])) or "-",
                 "modalités": " | ".join(map(str, rec["values"] or ["-"]))})
print(pd.DataFrame(rows).to_string(index=False))

                  item  classe  K         filtre                                               posé si                                               modalités
                region   GIVEN  3              -                                                     -                                 North | Central | South
           urban_rural   GIVEN  2              -                                                     -                                           Urban | Rural
              age_band   GIVEN  4              -                                                     -                             18-29 | 30-44 | 45-59 | 60+
        household_size   GIVEN  4              -                                                     -                               1 | 2-3 | 4-5 | 6 or more
household_has_children   GIVEN  2              -                                                     -                                                Yes | No
      has_mobile_phone   GIVEN  2             

`K` est le nombre de modalités de l'item, **sentinelle de filtre comprise** : un
item filtré a une case de plus qu'il n'a de réponses, parce que « jamais posé »
est pour lui une réponse à part entière. Cette case supplémentaire est la
dernière de votre vecteur de probabilités, et c'est de `K` qu'est calculée la
référence uniforme `U`.

`would_return` est filtré par `clinic_wait`, lui-même filtré par
`visited_clinic` : une chaîne de profondeur deux. Un répondant qui n'a jamais
consulté n'a jamais eu à dire combien de temps il avait attendu, ni s'il y
retournerait. Sa vraie réponse aux deux est `NA_GATED`.

In [4]:
chain = ["visited_clinic", "clinic_wait", "would_return"]
print(respondents[chain].value_counts().to_frame("répondants").head(12).to_string())

                                                         répondants
visited_clinic       clinic_wait           would_return            
No                   NA_GATED              NA_GATED            5179
Prefer not to answer NA_GATED              NA_GATED            4413
Yes                  Over 2 hours          Yes                 1405
                     Under 30 minutes      Yes                  380
                     Over 2 hours          No                   204
                                           Not sure             201
                     Under 30 minutes      Not sure             162
                                           No                    28
                     30 minutes to 2 hours Yes                   21
                                           Not sure               4
                                           No                     3


Lisez ce tableau comme la logique de filtre elle-même : partout où
`visited_clinic` vaut autre chose que `Yes`, les deux items enfants valent
`NA_GATED`, sans exception. **La réponse d'un item filtré est déterminée dès
que son parent est visible** — c'est du score gratuit, et la première chose à
exploiter.

### Ce que reçoit `predict()`

`score.py` prend les rôles visibles et masqués de la phase, empile les
répondants visibles au-dessus des répondants masqués, et vide chaque case
`PREDICT` de ces derniers. Ce sont ces cases vides pour lesquelles vous
renvoyez des probabilités.

`NaN` ne veut dire qu'une seule chose : *cette case est masquée, prédisez-la*.
Cela ne veut jamais dire « la personne n'a pas répondu » — une véritable
non-réponse est une modalité ordinaire comme `Prefer not to answer`, qui figure
dans la liste des modalités comme n'importe quelle autre.

In [5]:
frame, cells, truth = sample_rows(sample, respondents, PHASE)
shown = [n for n, r in sample["items"].items() if r["class"] != "EXCLUDE"]

print("table :", frame.shape, " cases à prédire :", len(cells))
print()
print(pd.concat([frame[["respondent_id"] + shown].head(3),
                 frame[["respondent_id"] + shown].tail(3)]).to_string(index=False))

table : (9900, 11)  cases à prédire : 7600

respondent_id  region urban_rural age_band household_size household_has_children has_mobile_phone visited_clinic  clinic_wait would_return trusts_health_advice
      R000001 Central       Rural      60+            4-5                    Yes               No            Yes Over 2 hours          Yes             Somewhat
      R000002   North       Rural      60+            4-5                    Yes               No             No     NA_GATED     NA_GATED           Not at all
      R000003   South       Rural      60+            2-3                     No               No             No     NA_GATED     NA_GATED                A lot
      R009898   South       Rural      60+            2-3                    Yes               No            NaN          NaN          NaN                  NaN
      R009899   South       Rural      60+            4-5                     No              Yes            NaN          NaN          NaN                  

Les premières lignes sont les répondants visibles : complets, et à vous d'en
apprendre. Les dernières lignes sont masquées — vous voyez leur bloc `GIVEN` et
rien d'autre.

Votre valeur de retour est un vecteur de probabilités par case vide, dans
l'**ordre canonique** : les lignes de haut en bas et, au sein d'une ligne, les
items dans l'ordre des clés de `schema["items"]` — et non l'ordre de
`frame.columns`, qui peut différer. Chaque vecteur suit les `values` de l'item
dans l'ordre, plus la case sentinelle lorsque l'item est filtré. Lisez l'ordre
dans le schéma, jamais dans les données : une modalité que personne n'a choisie
occupe quand même une case.

In [6]:
print(pd.DataFrame(cells, columns=["ligne", "respondent_id", "item"]).head(8)
      .to_string(index=False))

 ligne respondent_id                 item
  8000       R008001       visited_clinic
  8000       R008001          clinic_wait
  8000       R008001         would_return
  8000       R008001 trusts_health_advice
  8001       R008002       visited_clinic
  8001       R008002          clinic_wait
  8001       R008002         would_return
  8001       R008002 trusts_health_advice


### Notation

La référence de foule : chaque répondant masqué reçoit les parts lissées de
chaque item, sans rien tenir compte de l'individu. `skill` vaut 0 pour une
réponse uniforme et 1 pour la perfection, et c'est ce que classe le tableau des
scores.

In [7]:
def hidden_cells(frame, items):
    '''Chaque case vide, dans l'ordre où predict() doit les renvoyer.'''
    values = frame[items].to_numpy(dtype=object)
    ids = frame["respondent_id"].to_numpy(dtype=object)
    return [(row, ids[row], items[col])
            for row in range(values.shape[0])
            for col in range(len(items))
            if pd.isna(values[row, col])]


def crowd_for(sch, frame):
    items = generated_items(sch)
    tables = {}
    for item in items:
        counts = frame[item].value_counts()
        n = np.array([counts.get(o, 0) for o in options_for(sch, item)], float)
        tables[item] = (n + 0.5) / (n + 0.5).sum()
    return [tables[item] for _, _, item in hidden_cells(frame, items)]


vectors = floored(crowd_for(sample, frame), sample, cells, config["scoring"]["floor"])
result = grade(sample, config, vectors, truth, cells)

def table(rows):
    """Print label/value pairs, aligned however long the labels happen to be."""
    pad = max(len(label) for label, _ in rows)
    for label, value in rows:
        print("%-*s  %s" % (pad, label, value))


table([("référence uniforme (nats)", "%.4f" % result["uniform_reference"]),
       ("log-score", "%.4f" % result["log_score"]),
       ("skill", "%.4f" % result["skill"])])
print()
print("skill vaut 0 pour une réponse uniforme et 1 pour la perfection.")

référence uniforme (nats)  1.3144
log-score                  -0.8925
skill                      0.3210

skill vaut 0 pour une réponse uniforme et 1 pour la perfection.


Voilà tout le contrat. Une soumission est un `main.py` doté d'un `predict()`
qui renvoie ces vecteurs ; `score.py` l'exécute comme le fera l'évaluateur, et
`tools/check_submission_zip.py` vérifie que l'archive que vous téléversez est
bien formée.

---

## 2. Ce qu'apporte un bon modèle

Un benchmark qui récompense la prédiction des réponses des gens soulève une
inquiétude évidente : s'agit-il de cesser de les interroger ? Nous voyons ici une façon de combiner des prédictions algorithmiques et des échantillons humains.

Nous voulons un seul chiffre sur une population : la part des ménages qui font confiance aux conseils de santé de leur dispensaire. Le bloc bon marché — région, urbain ou rural, taille du ménage, présence d'un téléphone
— est déjà connu pour chaque ménage de la base de sondage, à partir de sources
administratives ou d'une enquête antérieure. Le bloc coûteux exige un enquêteur
à la porte, et le budget paie quelques centaines d'entretiens.

Vous avez trois options.

1. **Entretiens seuls.** Interroger 300 ménages, prendre la part, publier un
   intervalle de confiance. Valide, et aussi précis que 300 entretiens le
   permettent.
2. **Modèle seul.** Faire tourner un modèle sur le bloc bon marché pour chaque
   ménage et publier la moyenne. Gratuit, et *faux de tout ce dont le modèle se
   trompe* — sans intervalle, et sans aucun moyen de le savoir.
3. **Les deux.** Utiliser le modèle partout, puis se servir des 300 entretiens
   pour mesurer et retrancher l'erreur du modèle. C'est l'**inférence assistée
   par la prédiction**, et c'est ce que construit la suite de cette section.

La troisième est celle qui vaut la peine, parce qu'elle est valide que le
modèle soit bon ou mauvais, et *plus précise que la première quand le modèle
est bon*.

In [8]:
# L'estimande : la part de ceux qui font au moins un peu confiance aux conseils de santé.
TARGET, POSITIVE = "trusts_health_advice", ["Somewhat", "A lot"]
GIVEN = [n for n, r in sample["items"].items() if r["class"] == "GIVEN"]

Y = respondents[TARGET].isin(POSITIVE).to_numpy(float)
design = pd.get_dummies(respondents[GIVEN].astype(str), drop_first=True)
X = np.column_stack([np.ones(len(design)), design.to_numpy(float)])

# Le modèle est ajusté sur des répondants de vagues antérieures --
# le rôle TRAIN, c'est-à-dire exactement le bloc visible dont apprend une
# soumission. Il ne voit jamais les ménages que nous allons interroger.
past = (respondents[ROLE_COLUMN] == "TRAIN").to_numpy()
ridge = np.linalg.solve(X[past].T @ X[past] + 5 * np.eye(X.shape[1]),
                        X[past].T @ Y[past])
predicted = X @ ridge          # f(bloc bon marché), pour chaque ménage

# La base dont nous voulons un chiffre : les ménages non utilisés pour ajuster le modèle.
frame_rows = np.flatnonzero(~past)
TRUTH = Y[frame_rows].mean()   # connu seulement parce que les données sont inventées

table([("modèle ajusté sur, répondants antérieurs :", "%d" % past.sum()),
       ("base à estimer, ménages :", "%d" % len(frame_rows)),
       ("corrélation entre prédiction et réponse :", "%.2f"
        % np.corrcoef(predicted[frame_rows], Y[frame_rows])[0, 1]),
       ("part réelle (qu'une vraie enquête ne voit jamais) :", "%.3f" % TRUTH)])

modèle ajusté sur, répondants antérieurs :           8000
base à estimer, ménages :                            4000
corrélation entre prédiction et réponse :            0.58
part réelle (qu'une vraie enquête ne voit jamais) :  0.594


Tirons maintenant les 300 entretiens et calculons les trois chiffres.

L'intervalle « entretiens seuls » est celui des manuels. L'intervalle assisté
par la prédiction est la moyenne du modèle sur les ménages que vous n'avez
**pas** interrogés, corrigée par l'erreur moyenne du modèle sur ceux que vous
avez interrogés :

```
estimation = moyenne(prédiction | non interrogés) - [ moyenne(prédiction | interrogés) - moyenne(réponse | interrogés) ]
                    ↑ le modèle, utilisé partout          ↑ l'erreur du modèle, mesurée
```

Ce crochet constitue tout le mécanisme de sécurité. Il se calcule à partir de
réponses réelles, il coûte donc de vrais entretiens, et il élimine le biais du
modèle quel que soit ce biais.

In [9]:
def estimates(f, interviewed, rest):
    '''Estimations par entretiens seuls et assistée par la prédiction, chacune avec son erreur type.'''
    y = Y[interviewed]
    classical = (y.mean(), y.std(ddof=1) / np.sqrt(len(y)))

    correction = f[interviewed].mean() - y.mean()
    powered = (f[rest].mean() - correction,
               np.sqrt(f[rest].var(ddof=1) / len(rest)
                       + (f[interviewed] - y).var(ddof=1) / len(interviewed)))
    return classical, powered


def band(estimate):
    point, se = estimate
    return "%.3f  [%.3f, %.3f]  largeur %.3f" % (
        point, point - 1.96 * se, point + 1.96 * se, 2 * 1.96 * se)


N_INTERVIEWS = 300
draw = np.random.default_rng(SEED).permutation(frame_rows)
interviewed, rest = draw[:N_INTERVIEWS], draw[N_INTERVIEWS:]

classical, powered = estimates(predicted, interviewed, rest)
table([("vérité", "%.3f" % TRUTH),
       ("entretiens seuls", band(classical)),
       ("assistée par la prédiction", band(powered)),
       ("modèle seul (sans entretiens)", "%.3f  [aucun intervalle]"
        % predicted[frame_rows].mean())])

vérité                         0.594
entretiens seuls               0.597  [0.541, 0.652]  largeur 0.111
assistée par la prédiction     0.587  [0.541, 0.632]  largeur 0.091
modèle seul (sans entretiens)  0.590  [aucun intervalle]


Un seul tirage ne prouve rien — l'intervalle a pu être chanceux. Ce qui compte,
c'est le comportement sur de nombreuses enquêtes : l'intervalle contient-il la
vérité environ 95 % du temps, et quelle est sa largeur ? Répétons l'exercice
mille fois, chaque fois avec 300 nouveaux ménages.

In [10]:
def repeat(f, n_interviews=N_INTERVIEWS, draws=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(draws):
        shuffled = rng.permutation(frame_rows)
        classical, powered = estimates(f, shuffled[:n_interviews],
                                       shuffled[n_interviews:])
        out.append(classical + powered)
    return np.array(out)          # estimation, erreur type, estimation, erreur type


def summarize(trials, label):
    rows = []
    for name, point, se in (("entretiens seuls", trials[:, 0], trials[:, 1]),
                            ("assistée par la prédiction", trials[:, 2], trials[:, 3])):
        rows.append({"méthode": name,
                     "largeur moyenne": (2 * 1.96 * se).mean(),
                     "contient la vérité": np.mean(np.abs(point - TRUTH) <= 1.96 * se)})
    out = pd.DataFrame(rows)
    print(label)
    print(out.to_string(index=False, float_format="%.3f"))
    return out


good = summarize(repeat(predicted), "un modèle qui prédit bien")
narrower = 1 - good.loc[1, "largeur moyenne"] / good.loc[0, "largeur moyenne"]
print("\nla bande est %.0f %% plus étroite, à partir des mêmes %d entretiens." % (100 * narrower, N_INTERVIEWS))
print("pour acheter cette précision avec des entretiens seuls, il en faudrait environ %d." % round(N_INTERVIEWS / (1 - narrower) ** 2))

un modèle qui prédit bien
                   méthode  largeur moyenne  contient la vérité
          entretiens seuls            0.111               0.942
assistée par la prédiction            0.093               0.958

la bande est 17 % plus étroite, à partir des mêmes 300 entretiens.
pour acheter cette précision avec des entretiens seuls, il en faudrait environ 432.


Les deux intervalles contiennent la vérité environ 95 % du temps — c'est ce qui
en fait des intervalles. Celui assisté par la prédiction est simplement **plus
étroit**, à partir exactement du même travail de terrain. Lisez la dernière
ligne comme la morale de l'exercice : un meilleur modèle ne retire pas des
entretiens du budget, il fait compter davantage chacun d'eux.

### Ce qui se passe quand le modèle est mauvais

L'objection évidente est que cela ne marche que tant que le modèle a raison, et
que lui faire confiance est le risque. Voici la même procédure avec un modèle
ajusté sur une **population différente**.

In [11]:
from make_sandbox import make_sandbox

elsewhere = make_sandbox(sample, config, seed=99)      # une population différente
other_design = pd.get_dummies(elsewhere[GIVEN].astype(str), drop_first=True)
other_X = np.column_stack([np.ones(len(other_design)), other_design.to_numpy(float)])
other_Y = elsewhere[TARGET].isin(POSITIVE).to_numpy(float)

wrong = np.linalg.solve(other_X.T @ other_X + 5 * np.eye(other_X.shape[1]),
                        other_X.T @ other_Y)
mispredicted = X @ wrong

table([("corrélation entre prédiction et réponse :", "%.2f"
        % np.corrcoef(mispredicted[frame_rows], Y[frame_rows])[0, 1]),
       ("modèle seul (sans entretiens)", "%.3f   contre une vérité de %.3f   <- écart de %+.3f"
        % (mispredicted[frame_rows].mean(), TRUTH,
           mispredicted[frame_rows].mean() - TRUTH))])
print()
summarize(repeat(mispredicted), "un modèle qui ne se transfère pas")

corrélation entre prédiction et réponse :  0.06
modèle seul (sans entretiens)              0.727   contre une vérité de 0.594   <- écart de +0.133



un modèle qui ne se transfère pas
                   méthode  largeur moyenne  contient la vérité
          entretiens seuls            0.111               0.942
assistée par la prédiction            0.117               0.944


,méthode,largeur moyenne,contient la vérité
0,entretiens seuls,0.111157,0.942
1,assistée par la prédiction,0.116888,0.944


Observez : l'estimation par modèle seul se trompe de plus d'un dixième, et rien
dans la sortie ne vous l'aurait dit — pas d'intervalle, pas d'avertissement,
juste un nombre qui a l'air exactement aussi autorisé que le bon. Remplacer le
terrain par le modèle introduit un biais.

L'intervalle assisté par la prédiction contient toujours la vérité environ 95 %
du temps. Il n'est pas plus étroit que les entretiens seuls — un modèle inutile
n'achète aucune précision. Le terme de correction a mesuré l'erreur du modèle
sur les 300 entretiens réels et l'a retranchée, ce pour quoi il est là.

### Pourquoi cela demande un benchmark

La largeur de cette bande est une fonction directe de la qualité du modèle.
D'où l'intérêt de mesurer soigneusement la qualité de prédiction, sur de vrais
instruments, avec une règle de score propre : un meilleur `skill` au classement,
c'est un intervalle de confiance plus étroit sur le terrain, ou le même
intervalle avec moins d'entretiens.